# Hunt-and-Kill y Búsqueda no Informada

**Curso:** Inteligencia Artificial · **Referencia:** AIMA, capítulo 3  
**Modalidad:** individual · **Duración sugerida:** 3 semanas  
**Producto:** un cuaderno Jupyter con tres implementaciones comparables

## Problema

Se proporciona solamente un generador de laberintos perfectos mediante **Hunt-and-Kill**. A partir del grafo producido, el estudiante deberá diseñar, implementar, verificar y comparar algoritmos de búsqueda no informada para encontrar un camino entre dos celdas.

El proyecto debe realizarse en tres versiones:

1. **Desde cero:** sin bibliotecas de búsqueda ni de grafos.
2. **SimpleAI:** modelando el laberinto como un `SearchProblem`.
3. **AIMA-Python:** modelándolo como una subclase de `Problem`.

> Este cuaderno es deliberadamente incompleto. No contiene implementaciones Python de DFS, BFS, UCS, IDDFS, búsqueda bidireccional ni Lee. El trabajo evaluado consiste en convertir las especificaciones y el pseudocódigo en soluciones propias.

## Resultados de aprendizaje

Al finalizar, el estudiante estará en capacidad de:

- formular un entorno como espacio de estados;
- separar el problema de la estrategia de búsqueda;
- implementar búsqueda en árbol y búsqueda en grafo;
- justificar completitud, optimalidad y complejidad;
- adaptar un mismo problema a dos bibliotecas de IA;
- construir experimentos reproducibles;
- interpretar el algoritmo de Lee como BFS por frentes de onda;
- defender decisiones de diseño y diagnosticar errores sin depender de una solución externa.

## Condiciones y restricciones

- No se permite `networkx` ni funciones externas que ya resuelvan caminos.
- En la versión desde cero no se permite SimpleAI, AIMA-Python ni otra biblioteca de búsqueda.
- Los estados deben ser inmutables y utilizables como claves de diccionario.
- Cada algoritmo debe devolver solución y métricas, no solamente imprimirlas.
- Todas las versiones deben trabajar con el **mismo grafo**, inicio, meta y costos.
- Los resultados deben ser reproducibles mediante semillas.
- No se acepta una imagen como evidencia: deben entregarse estructuras de datos verificables.
- El estudiante debe citar cualquier recurso consultado y conservar una bitácora breve de decisiones.

### Evidencia de autoría y comprensión

La evaluación incluye:

1. historial de al menos seis hitos o commits;
2. una traza manual sobre un laberinto pequeño;
3. defensa oral individual de 5–8 minutos;
4. modificación en vivo de inicio, meta, semilla o costo;
5. explicación de una decisión que produjo un error y cómo se corrigió.

Durante la defensa se podrá solicitar reconstruir una parte pequeña del algoritmo sin consultar el cuaderno.

## 1. Código suministrado: generador Hunt-and-Kill

Este es el único algoritmo completo entregado. El resultado es un diccionario de adyacencia no dirigido:

```text
grafo[celda] = conjunto de celdas conectadas por un corredor
```

Una celda se representa como `(fila, columna)`.

In [ ]:
import random


class LaberintoHuntKill:
    """Genera un laberinto perfecto como grafo no dirigido."""

    DIRECCIONES = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self, filas=25, columnas=25, semilla=2026):
        self.filas = filas
        self.columnas = columnas
        self.rng = random.Random(semilla)
        self.grafo = {
            (fila, columna): set()
            for fila in range(filas)
            for columna in range(columnas)
        }

    def vecinos_geometricos(self, celda):
        fila, columna = celda
        for df, dc in self.DIRECCIONES:
            vecino = (fila + df, columna + dc)
            if (0 <= vecino[0] < self.filas and
                    0 <= vecino[1] < self.columnas):
                yield vecino

    def conectar(self, origen, destino):
        self.grafo[origen].add(destino)
        self.grafo[destino].add(origen)

    def generar(self):
        no_visitadas = set(self.grafo)
        actual = self.rng.choice(tuple(no_visitadas))
        no_visitadas.remove(actual)

        while no_visitadas:
            # KILL: avanzar aleatoriamente hacia una celda no visitada.
            libres = [
                vecino
                for vecino in self.vecinos_geometricos(actual)
                if vecino in no_visitadas
            ]

            if libres:
                siguiente = self.rng.choice(libres)
                self.conectar(actual, siguiente)
                no_visitadas.remove(siguiente)
                actual = siguiente
                continue

            # HUNT: localizar una celda libre vecina del árbol construido.
            candidatas = []
            for celda in no_visitadas:
                visitados = [
                    vecino
                    for vecino in self.vecinos_geometricos(celda)
                    if vecino not in no_visitadas
                ]
                if visitados:
                    candidatas.append((celda, visitados))

            actual, visitados = self.rng.choice(candidatas)
            vecino = self.rng.choice(visitados)
            self.conectar(actual, vecino)
            no_visitadas.remove(actual)

        return self.grafo

### Actividad 1 — Auditoría del generador

Antes de resolver el laberinto:

1. Explique las fases Hunt y Kill señalando las instrucciones correspondientes.
2. Demuestre que el grafo generado es conexo.
3. Justifique por qué contiene exactamente $|V|-1$ aristas.
4. Explique por qué esas dos propiedades implican que es un árbol.
5. Diseñe pruebas para comprobar:
   - simetría de las adyacencias;
   - ausencia de conexiones diagonales;
   - validez de coordenadas;
   - conectividad;
   - ausencia de ciclos;
   - reproducibilidad con una semilla.

No basta con ejecutar el generador: las propiedades deben verificarse automáticamente.

**Respuesta y pruebas del estudiante:**

### 1. Fases Hunt y Kill

*(referidas a las líneas del código anterior)*

**KILL** — bloque comentado `# KILL: avanzar aleatoriamente...`

Mientras la celda `actual` tenga al menos un vecino geométrico que todavía esté en `no_visitadas`, el algoritmo se comporta como una **caminata aleatoria autoevitante**:

1. Elige uno de esos vecinos libres (`self.rng.choice(libres)`).
2. Lo conecta a `actual` (`self.conectar`).
3. Lo marca como visitado y lo convierte en la nueva celda `actual` (`continue` vuelve al inicio del `while`).

Esta fase construye "pasillos" en línea recta mientras hay territorio virgen alcanzable directamente.

**HUNT** — bloque comentado `# HUNT: localizar una celda libre...`

Se activa solo cuando KILL se atasca, es decir, cuando `actual` no tiene ningún vecino geométrico libre (`libres` queda vacío). Entonces:

1. Se recorren **todas** las celdas aún no visitadas (`for celda in no_visitadas`).
2. Para cada una, se listan sus vecinos que **ya** fueron visitados (`visitados`).
3. Cualquier celda con al menos un vecino visitado es candidata a "resucitar" la caminata.
4. Se elige una candidata al azar, se conecta con uno de sus vecinos ya visitados y se retoma como `actual`.

Esta fase es la que evita que el algoritmo quede atrapado: garantiza que siempre se pueda seguir creciendo el árbol mientras existan celdas sin visitar.



### 2. El grafo generado es conexo

Por construcción, toda arista que se agrega (`self.conectar`) une una celda que **ya** pertenece al árbol parcial (visitada, ergo alcanzable desde la raíz `actual` inicial) con una celda recién retirada de `no_visitadas`:

- En la fase **KILL**, `siguiente` se conecta a `actual` (visitada).
- En la fase **HUNT**, `actual` (la candidata, recién retirada de `no_visitadas`) se conecta con `vecino` (ya visitado).

Es decir, cada celda, en el momento en que deja de estar en `no_visitadas`, queda unida por una arista a una celda que ya era alcanzable desde la primera celda elegida.

> **Por inducción sobre el orden de remoción de `no_visitadas`:**
> la primera celda es trivialmente alcanzable (es la raíz); si las primeras *k* celdas removidas son alcanzables entre sí, la celda *k+1* se conecta a una de ellas, luego también es alcanzable.

Como el bucle `while no_visitadas` solo termina cuando el conjunto queda vacío, todas las celdas terminan siendo alcanzables desde la raíz: **el grafo es conexo**.



### 3. El grafo tiene exactamente |V| − 1 aristas

`self.conectar` es la **única** función que agrega aristas, y se invoca exactamente una vez por iteración exitosa del bucle exterior (una vez en KILL, una vez en HUNT, nunca ambas en la misma vuelta).

Cada llamada a `conectar` remueve exactamente una celda de `no_visitadas` (`no_visitadas.remove(...)`), y nunca se vuelve a insertar una celda ya removida (no hay `add` sobre `no_visitadas`).

Como `no_visitadas` arranca con |V| − 1 elementos (la raíz se remueve antes del bucle) y termina en 0, el bucle ejecuta exactamente |V| − 1 remociones, y por lo tanto exactamente **|V| − 1 llamadas a `conectar`**, es decir, |V| − 1 aristas no dirigidas.



### 4. Conexo + |V| − 1 aristas implica árbol

Es un resultado estándar de teoría de grafos (AIMA, cap. 3, y cualquier texto de grafos):

> Un grafo simple, no dirigido y conexo con *n* vértices es un árbol **si y solo si** tiene exactamente *n* − 1 aristas.

**Intuición:**

- Si tuviera **menos** de *n* − 1 aristas, no podría ser conexo (se necesitan al menos *n* − 1 aristas para conectar *n* vértices).
- Si teniendo *n* vértices conexos tuviera **más** de *n* − 1 aristas, existiría al menos una arista "sobrante" cuya eliminación no desconecta el grafo, lo cual implica que esa arista cerraba un ciclo.

Como el generador produce **exactamente** *n* − 1 aristas (punto 3) y el grafo es conexo (punto 2), no puede contener ciclos: es, por definición, un **árbol** (de ahí que se hable de *"laberinto perfecto"*: un único camino simple entre cualquier par de celdas).

In [ ]:
import unittest

class TestGeneradorHuntKill(unittest.TestCase):
 
    def setUp(self):
        self.filas, self.columnas = 8, 8
        self.gen = LaberintoHuntKill(
            filas=self.filas, columnas=self.columnas, semilla=42
        )
        self.grafo = self.gen.generar()
        self.n_celdas = self.filas * self.columnas
 
    # simetría 
    def test_simetria_de_adyacencias(self):
        for celda, vecinos in self.grafo.items():
            for vecino in vecinos:
                self.assertIn(
                    celda, self.grafo[vecino],
                    f"{celda} -> {vecino} no es simétrica",
                )
 
    # sin diagonales 
    def test_sin_conexiones_diagonales(self):
        for (f1, c1), vecinos in self.grafo.items():
            for (f2, c2) in vecinos:
                dist_fila = abs(f1 - f2)
                dist_col = abs(c1 - c2)
                # Debe ser ortogonal: exactamente un paso en una sola dimensión (Manhattan = 1), nunca diagonal.
                self.assertEqual(dist_fila + dist_col, 1)
 
    # coordenadas válidas 
    def test_coordenadas_validas(self):
        for celda in self.grafo:
            fila, columna = celda
            self.assertTrue(0 <= fila < self.filas)
            self.assertTrue(0 <= columna < self.columnas)
        for vecinos in self.grafo.values():
            for (fila, columna) in vecinos:
                self.assertTrue(0 <= fila < self.filas)
                self.assertTrue(0 <= columna < self.columnas)
 
    # conectividad (BFS/DFS propio, sin bibliotecas) 
    def test_conectividad(self):
        raiz = next(iter(self.grafo))
        visitados = {raiz}
        pila = [raiz]
        while pila:
            actual = pila.pop()
            for vecino in self.grafo[actual]:
                if vecino not in visitados:
                    visitados.add(vecino)
                    pila.append(vecino)
        self.assertEqual(len(visitados), self.n_celdas)
 
    # ausencia de ciclos (número de aristas == |V| - 1) 
    def test_ausencia_de_ciclos(self):
        num_aristas = sum(len(v) for v in self.grafo.values()) // 2
        self.assertEqual(num_aristas, self.n_celdas - 1)
 
    def test_ausencia_de_ciclos_por_deteccion_directa(self):

        raiz = next(iter(self.grafo))
        visitados = set()
 
        def dfs(nodo, padre):
            visitados.add(nodo)
            for vecino in self.grafo[nodo]:
                if vecino == padre:
                    continue
                if vecino in visitados:
                    self.fail(f"Ciclo detectado en torno a {vecino}")
                dfs(vecino, nodo)
 
        dfs(raiz, None)
        self.assertEqual(len(visitados), self.n_celdas)
 
    # reproducibilidad 
    def test_reproducibilidad_misma_semilla(self):
        gen_a = LaberintoHuntKill(filas=8, columnas=8, semilla=123)
        gen_b = LaberintoHuntKill(filas=8, columnas=8, semilla=123)
        grafo_a = gen_a.generar()
        grafo_b = gen_b.generar()
        self.assertEqual(grafo_a, grafo_b)
 
    def test_semillas_distintas_pueden_diferir(self):
        gen_a = LaberintoHuntKill(filas=8, columnas=8, semilla=1)
        gen_b = LaberintoHuntKill(filas=8, columnas=8, semilla=2)
        grafo_a = gen_a.generar()
        grafo_b = gen_b.generar()
        # No es una garantía matemática absoluta, pero con 8x8 celda la probabilidad de colisión exacta es despreciable
        self.assertNotEqual(grafo_a, grafo_b)
suite = unittest.TestLoader().loadTestsFromTestCase(TestGeneradorHuntKill)
unittest.TextTestRunner(verbosity=2).run(suite)

## 2. Instancia individual

Cada estudiante debe construir una instancia reproducible:

- `semilla = últimos 5 dígitos del código estudiantil`;
- `filas = 20 + último dígito`;
- `columnas = 20 + penúltimo dígito`;
- inicio y meta deben estar en cuadrantes opuestos, pero no necesariamente en las esquinas.

Registre esos valores en una tabla. El docente podrá cambiar cualquiera de ellos durante la defensa.

| Parámetro | Valor |
|---|---|
| Semilla | 95022 |
| Filas | 22 |
| Columnas | 22 |
| Inicio | (21, 9) |
| Meta | (4, 12) |

In [ ]:
import random

semilla = 95022
filas = 20 + int(str(semilla)[-1])  
columnas = 20 + int(str(semilla)[-2])

generador = LaberintoHuntKill(filas=filas, columnas=columnas, semilla=semilla)
grafo = generador.generar()

# Elegir inicio y meta en cuadrantes opuestos (no necesariamente esquinas)
def cuadrante_de(celda, filas, columnas):
    fila, columna = celda
    mitad_fila, mitad_columna = filas / 2, columnas / 2
    if fila < mitad_fila and columna < mitad_columna:
        return 1
    if fila < mitad_fila and columna >= mitad_columna:
        return 2
    if fila >= mitad_fila and columna < mitad_columna:
        return 3
    return 4

cuadrante_opuesto = {1: 4, 2: 3, 3: 2, 4: 1}

# RNG independiente pero igual de reproducible (también derivado de la semilla).
rng_posiciones = random.Random(semilla + 1)
cuadrante_inicio = rng_posiciones.choice([1, 2, 3, 4])
cuadrante_meta = cuadrante_opuesto[cuadrante_inicio]

celdas_inicio = [c for c in grafo if cuadrante_de(c, filas, columnas) == cuadrante_inicio]
celdas_meta = [c for c in grafo if cuadrante_de(c, filas, columnas) == cuadrante_meta]

inicio = rng_posiciones.choice(celdas_inicio)
meta = rng_posiciones.choice(celdas_meta)

print("| Parámetro | Valor |")
print("|---|---|")
print(f"| Semilla   | {semilla} |")
print(f"| Filas     | {filas} |")
print(f"| Columnas  | {columnas} |")
print(f"| Inicio    | {inicio} |")
print(f"| Meta      | {meta} |")

## 3. Formulación formal del problema

Defina rigurosamente:

| Componente | Especificación por completar |
|---|---|
| Conjunto de estados       | Todas las celdas (fila, columna) con 0 <= fila < filas, 0 <= columna < columnas. Un estado es una tupla inmutable (fila, columna); coincide con los vértices del grafo generado por LaberintoHuntKill. |
| Estado inicial            | La celda `inicio` fijada en la instancia individual (sección 2).              |
| Acciones aplicables       | Dado un estado s, Acciones(s) = { mover_a(v) : v ∈ grafo[s] }. Es decir, moverse a cualquier celda directamente conectada a s por un corredor. El conjunto depende solo de las adyacencias del grafo, no de la posición geométrica de s en la cuadrícula. |
| Modelo de transición      | Resultado(s, mover_a(v)) = v, para v ∈ grafo[s]. Determinista: cada acción produce exactamente un sucesor. |
| Prueba de objetivo        | EsMeta(s)  ⇔  s = meta.                                                       |
| Costo de paso             | En la versión base (laberinto perfecto, sin ponderar): costo(s, a, s') = 1 para toda transición válida. En la extensión de la sección 9.B, costo(s, a, s') = peso(arista(s, s')) > 0, definido explícitamente por arista (ver esa sección). |
| Costo de una solución     | Suma de los costos de paso a lo largo del camino inicio → meta: g(meta) = Σ costo(sᵢ, aᵢ, sᵢ₊₁) para i = 0 .. n-1, donde s₀ = inicio y sₙ = meta. ||

Además:

1. diferencie **estado**, **nodo de búsqueda** y **celda dibujada**;
2. indique cuándo dos nodos distintos pueden contener el mismo estado;
3. explique por qué debe usarse búsqueda en grafo;
4. establezca invariantes que deben cumplirse durante una búsqueda.

## Respuesta a preguntas adicionales

### 1. Estado vs. nodo de búsqueda vs. celda dibujada

- **Estado:** la tupla `(fila, columna)` tal como aparece en el grafo. Es la unidad **matemática** del problema; es inmutable y hasheable.

- **Nodo de búsqueda:** estructura auxiliar utilizada por el algoritmo que **envuelve un estado** y añade metadatos de la búsqueda:
  - Padre.
  - Acción que lo generó.
  - Costo acumulado `g`.
  - Profundidad.

  Varios nodos de búsqueda distintos pueden envolver el mismo estado (ver pregunta 2).

- **Celda dibujada:** representación gráfica o visual del estado, por ejemplo, un píxel o un rectángulo coloreado en una figura de `matplotlib`. Se utiliza únicamente para mostrar el resultado a una persona.

  La celda dibujada **no participa en el algoritmo** y no debe utilizarse como fuente de verdad para ninguna prueba automática. De ahí la restricción de la sección **«Condiciones y restricciones»**: *«no se acepta una imagen como evidencia»*.

---

### 2. ¿Cuándo dos nodos distintos contienen el mismo estado?

Esto ocurre cuando el grafo **no es un árbol**, es decir, cuando existe más de un camino posible entre el estado inicial y un mismo estado.

Por ejemplo, esto puede ocurrir después de introducir ciclos en la sección **9.A**. En ese caso:

1. Dos ramas distintas de la búsqueda pueden llegar al mismo estado.
2. Cada rama puede haber seguido un camino diferente.
3. Por lo tanto, los caminos pueden tener costos acumulados `g` diferentes.
4. Se generan dos nodos de búsqueda distintos, con diferente padre y/o diferente `g`, que envuelven el mismo estado.

En el **laberinto perfecto original (árbol)** esto no puede ocurrir en una búsqueda en árbol sin repetición, porque solo existe un camino simple entre cualquier par de celdas.

Aun así, el mismo estado puede aparecer **transitoriamente en la frontera** si es generado como sucesor de dos nodos distintos antes de ser expandido.

---

### 3. ¿Por qué debe usarse búsqueda en grafo?

La **búsqueda en árbol**, al no tener control de repetidos, vuelve a expandir estados ya visitados cada vez que estos reaparecen como sucesores.

Esto presenta dos problemas:

- **En el laberinto base:** es innecesario, porque el grafo es un árbol y cualquier repetición simplemente representaría volver por donde se vino.
- **Cuando existen ciclos (sección 9.A):** el problema puede volverse explosivo, ya que el algoritmo podría recorrer el mismo ciclo indefinidamente y no terminar en un tiempo razonable.

La **búsqueda en grafo**, mediante un conjunto de estados `alcanzados` o `visitados`:

- Evita procesar repetidamente los mismos estados.
- Evita trabajo redundante.
- Garantiza la terminación en los casos correspondientes, evitando recorridos infinitos por ciclos.

La contrapartida es un **mayor consumo de memoria**, ya que es necesario almacenar los estados que ya han sido vistos.

---

### 4. Invariantes que deben cumplirse durante una búsqueda

Durante la ejecución del algoritmo deben mantenerse, como mínimo, los siguientes invariantes:

- **Alcanzabilidad:** todo estado que aparece en la frontera debe ser alcanzable desde el estado inicial mediante una secuencia válida de acciones.

- **Consistencia del costo:** `g(nodo)` debe ser exactamente la suma de los costos de los pasos a lo largo del camino registrado mediante los punteros a los padres. No debe ser un valor inventado o estar desincronizado del camino que puede reconstruirse.

- **Estados resueltos:** un estado no debe estar simultáneamente marcado como **definitivamente resuelto** (por ejemplo, expandido con costo óptimo en UCS) y seguir generando reemplazos con un costo peor.

- **Contador de expansiones:** `expandidos` solo debe incrementarse cuando un nodo se retira de la frontera **y se generan sus sucesores**, de acuerdo con el contrato del punto 4. No debe incrementarse dos veces para el mismo nodo.

- **Reconstrucción del camino:** cuando `encontrado = True`, el camino reconstruido debe comenzar en `inicio` y terminar en `meta`. Cuando `encontrado = False`, el camino debe ser vacío.


## 4. Contrato común de las tres versiones

Todos los algoritmos deben entregar un registro equivalente a:

```text
ResultadoBusqueda
    encontrado: booleano
    camino: secuencia de estados
    acciones: secuencia de acciones
    costo: número
    profundidad: entero
    expandidos: entero
    generados: entero
    repetidos_descartados: entero
    frontera_maxima: entero
    tiempo_ms: número
```

### Convenciones obligatorias

- Un estado se cuenta como **expandido** al retirarlo de la frontera y generar sus sucesores.
- Un nodo meta retirado de la frontera no se cuenta como expandido si no se generan sucesores.
- **Generado** significa nodo sucesor construido, aunque luego sea descartado.
- La política para estados repetidos debe documentarse.
- El camino debe incluir inicio y meta.
- Si no hay solución, `camino` debe ser vacío y `encontrado` falso.

Estas convenciones son necesarias para comparar implementaciones de manera justa.

## 5. Pseudocódigo base: búsqueda en grafo

El siguiente esquema no especifica la estructura de la frontera ni resuelve el manejo de costos. Debe especializarlo y justificar cada decisión.

```text
BUSQUEDA-GRAFO(problema, frontera):
    nodo_inicial ← crear_nodo(problema.estado_inicial)
    insertar(frente, nodo_inicial)
    alcanzados ← estructura apropiada

    mientras frontera no esté vacía:
        nodo ← extraer_segun_politica(frontera)

        si ES_META(nodo.estado):
            devolver RECONSTRUIR(nodo)

        si nodo debe expandirse:
            para cada acción aplicable:
                hijo ← construir sucesor
                decidir si insertar, reemplazar o descartar hijo

        actualizar métricas

    devolver FRACASO
```

### Preguntas de diseño

1. ¿Cuándo debe marcarse un estado: al generarlo o al expandirlo?
2. ¿La respuesta cambia entre BFS, DFS y UCS?
3. ¿Qué información mínima debe almacenar cada nodo?
4. ¿Cómo se reconstruye el camino sin copiar listas completas en cada nodo?
5. ¿Cómo se resuelve un camino más barato hacia un estado ya descubierto?

# Versión 1 — Implementación desde cero

No puede utilizar bibliotecas de búsqueda o grafos. Se permiten únicamente estructuras estándar como listas, diccionarios, conjuntos, `deque` y colas de prioridad.

Debe implementar:

1. DFS iterativa;
2. BFS;
3. búsqueda de costo uniforme;
4. búsqueda limitada en profundidad;
5. profundización iterativa;
6. búsqueda bidireccional;
7. Lee o expansión por frente de onda.

La versión debe separar, como mínimo:

- representación del problema;
- representación de nodos;
- política de frontera;
- control de repetidos;
- reconstrucción del camino;
- recolección de métricas.

## 6. Pseudocódigo que debe traducirse

### DFS y BFS

```text
inicializar frontera con el nodo inicial
inicializar alcanzados

mientras haya nodos:
    extraer según disciplina LIFO o FIFO
    comprobar objetivo
    expandir
    insertar estados nuevos
```

La diferencia no debe quedar dispersa por todo el programa. Diseñe una abstracción que permita cambiar la política de frontera.

### Costo uniforme

```text
frontera ← cola de prioridad ordenada por g(n)
mejor_costo[inicial] ← 0

mientras frontera no esté vacía:
    extraer nodo con menor g
    ignorar entradas obsoletas
    comprobar objetivo
    para cada sucesor:
        nuevo_costo ← g(nodo) + costo_de_paso
        si mejora el costo conocido:
            actualizar costo, padre y frontera
```

### Profundización iterativa

```text
para límite = 0, 1, 2, ...:
    resultado ← BUSQUEDA-LIMITADA(problema, límite)
    si resultado es solución: devolverlo
    si resultado es fracaso definitivo: terminar
```

Debe distinguir **corte** de **fracaso**.

### Bidireccional

```text
crear una frontera desde inicio y otra desde meta
expandir de forma equilibrada
detectar intersección entre regiones alcanzadas
unir los dos caminos respetando su orientación
```

No se acepta ejecutar dos búsquedas completas y comparar al final.

In [ ]:
# VERSIÓN 1: implemente aquí su solución desde cero.

## 7. Lee: propagación de un frente luminoso

No se modela un único rayo que rebota. Se modela una onda que avanza simultáneamente por todos los corredores accesibles.

```text
LEE(grafo, inicio, meta):
    etiqueta[inicio] ← 0
    frente_actual ← {inicio}

    mientras frente_actual no esté vacío y meta no tenga etiqueta:
        frente_siguiente ← vacío
        para cada celda del frente_actual:
            para cada vecino transitable sin etiqueta:
                etiqueta[vecino] ← etiqueta[celda] + 1
                registrar procedencia o dirección
                agregar vecino al frente_siguiente
        frente_actual ← frente_siguiente

    si meta no tiene etiqueta: devolver fracaso
    reconstruir desde meta siguiendo etiquetas decrecientes
```

### Requisitos adicionales

- Dibuje al menos ocho instantes del frente de onda.
- Use una escala de color que represente el tiempo de llegada.
- Compruebe que la etiqueta de la meta coincide con la profundidad de BFS.
- Explique formalmente por qué Lee es BFS en un grafo no ponderado.
- Analice qué deja de funcionar cuando los costos no son unitarios.

In [ ]:
# Implemente y visualice aquí el algoritmo de Lee.

# Versión 2 — SimpleAI

Instalación orientativa:

```text
pip install simpleai
```

Consulte la documentación oficial. Debe crear una subclase de `SearchProblem` y definir, como mínimo:

| Método de SimpleAI | Responsabilidad en el laberinto |
|---|---|
| `actions(state)` | Producir únicamente movimientos legales |
| `result(state, action)` | Calcular el estado sucesor sin mutar el original |
| `is_goal(state)` | Reconocer la celda meta |
| `cost(state, action, state2)` | Devolver el costo del corredor |

Instancie el problema con un estado inicial inmutable y use `graph_search=True`.

Debe ejecutar y estudiar:

- `depth_first`;
- `breadth_first`;
- `uniform_cost`;
- `limited_depth_first`;
- `iterative_limited_depth_first`.

### Exigencia avanzada

SimpleAI devuelve un nodo solución, pero las métricas del contrato no aparecen automáticamente con la misma definición. Investigue su mecanismo de *viewer* o diseñe instrumentación externa **sin modificar el código fuente instalado de la biblioteca**. Documente cualquier diferencia entre las métricas de SimpleAI y las de su versión.

No copie la estructura de su buscador desde cero dentro de esta versión: el propósito es construir correctamente el adaptador del problema y comprender la biblioteca.

In [ ]:
# VERSIÓN 2: modele el problema y ejecute SimpleAI.

# Versión 3 — AIMA-Python

Repositorio de referencia: `aimacode/aima-python`.

Debe modelar el laberinto como una subclase de `Problem`. Investigue y documente los contratos de:

| Método de AIMA-Python | Decisión requerida |
|---|---|
| `actions(state)` | Formato de las acciones y orden determinista |
| `result(state, action)` | Transición entre celdas |
| `goal_test(state)` | Prueba de objetivo |
| `path_cost(c, state1, action, state2)` | Acumulación de costos |

Compare al menos:

- `depth_first_graph_search`;
- `breadth_first_graph_search`;
- `uniform_cost_search`;
- `depth_limited_search`;
- `iterative_deepening_search`;
- `bidirectional_search`.

### Exigencia avanzada

Instrumente los algoritmos sin alterar permanentemente el repositorio. Puede utilizar una subclase del problema, envoltorios o contadores cuidadosamente definidos. Explique por qué contar llamadas a `actions` no siempre coincide con contar nodos generados.

Compare la representación del nodo de AIMA-Python con su propia clase de nodo.

In [ ]:
# VERSIÓN 3: adapte el problema a AIMA-Python.

## 8. Pruebas de aceptación

El estudiante debe implementar pruebas automáticas equivalentes a las siguientes especificaciones, sin recibir su solución:

### Validez de un camino

Para cada resultado exitoso:

1. el primer estado es el inicio;
2. el último estado es la meta;
3. cada pareja consecutiva aparece como arista en el grafo;
4. el costo reportado coincide con la suma de costos;
5. aplicar las acciones reproduce exactamente los estados.

### Concordancia entre versiones

- En costos unitarios: `longitud(BFS) = longitud(UCS) = etiqueta_meta(Lee)`.
- En un árbol: todas las soluciones válidas contienen la misma secuencia de estados.
- Con ciclos: `longitud(BFS) ≤ longitud(DFS)` para una DFS que encuentre solución.
- Con costos positivos: `costo(UCS) ≤ costo(cualquier solución BFS)`.
- Las tres versiones deben concordar en existencia de solución y costo óptimo.

### Casos límite obligatorios

- inicio igual a meta;
- laberinto de `1×1`;
- meta inalcanzable después de eliminar corredores;
- múltiples entradas obsoletas en UCS;
- dos posibles puntos de encuentro bidireccional;
- límite exacto, insuficiente y excesivo en búsqueda limitada.

In [ ]:
# Escriba aquí su batería de pruebas.

## 9. Segunda familia de problemas: ciclos y costos

Hunt-and-Kill produce un árbol. Allí existe un único camino entre dos estados y no se aprecia plenamente la optimalidad. Diseñe, sin modificar el generador original, dos transformaciones:

### A. Laberinto con ciclos

Abra aleatoriamente entre 5% y 12% de las paredes internas que no sean corredores. Debe conservar:

- adyacencias ortogonales;
- simetría del grafo;
- reproducibilidad;
- al menos un ciclo comprobable.

### B. Laberinto ponderado

Asigne costos positivos a corredores o terrenos. Construya deliberadamente un caso donde:

```text
camino con menos pasos ≠ camino de menor costo
```

No use pesos negativos. Justifique si los costos pertenecen a celdas, acciones o aristas y mantenga esa decisión en las tres versiones.

In [ ]:
# Implemente aquí ciclos y costos; no modifique la clase suministrada.

## 10. Diseño experimental

Ejecute como mínimo **30 instancias** por configuración y reporte mediana y rango intercuartílico, no solamente un caso.

| Factor | Niveles mínimos |
|---|---|
| Tamaño | 15×15, 25×25, 40×40 |
| Topología | árbol, 5% ciclos, 10% ciclos |
| Costos | unitarios, ponderados |
| Posición | esquinas, interior, cercanas |
| Algoritmos | todos los exigidos que sean aplicables |

Debe controlar semillas y separar tiempo de generación del tiempo de búsqueda.

### Gráficas obligatorias

1. expandidos frente a número de estados;
2. frontera máxima frente a profundidad de solución;
3. tiempo frente a tamaño;
4. costo y longitud de las soluciones;
5. mapa de calor de Lee;
6. comparación cruzada de las tres versiones.

### Discusión obligatoria

- ¿Qué métrica resulta más estable que el tiempo?
- ¿Cuándo la búsqueda bidireccional pierde su ventaja?
- ¿Por qué IDDFS repite trabajo y aun así puede ser conveniente?
- ¿Qué efecto tiene el orden de sucesores sobre DFS?
- ¿Qué parte de las diferencias se debe al algoritmo y cuál a la biblioteca?

In [ ]:
# Construya aquí su protocolo experimental, tablas y gráficas.

## 11. Análisis teórico

Complete y justifique esta tabla usando $b$ como factor de ramificación, $d$ como profundidad de la solución menos profunda y $m$ como profundidad máxima.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones |
|---|---|---|---|---|---|
| DFS | | | | | |
| BFS | | | | | |
| UCS | | | | | |
| DLS | | | | | |
| IDDFS | | | | | |
| Bidireccional | | | | | |
| Lee | | | | | |

Relacione después las cotas teóricas con las mediciones. Una tabla memorizada sin conexión con los experimentos no recibe puntaje completo.

## 12. Preguntas para la defensa

El docente seleccionará algunas al azar:

1. Muestre en memoria la diferencia entre frontera y alcanzados.
2. Cambie BFS a DFS modificando solamente la política apropiada.
3. Explique un caso en el que marcar visitados demasiado tarde duplique trabajo.
4. Explique por qué terminar UCS al generar la meta puede ser incorrecto.
5. Reconstruya un camino usando padres sin almacenar caminos completos.
6. Muestre por qué dos BFS bidireccionales requieren orientar correctamente los padres.
7. Explique por qué Lee no es un rayo geométrico.
8. Adapte el problema a una meta múltiple.
9. Introduzca una celda bloqueada durante la ejecución y analice qué debe recalcularse.
10. Compare `SearchProblem` de SimpleAI con `Problem` de AIMA-Python.
11. Diagnostique una discrepancia de métricas entre dos versiones.
12. Argumente por qué un laberinto perfecto oculta diferencias de calidad de ruta.

## 13. Entrega

El cuaderno final debe contener:

1. auditoría del generador;
2. formulación formal;
3. versión desde cero;
4. versión SimpleAI;
5. versión AIMA-Python;
6. pruebas automáticas;
7. extensión con ciclos y costos;
8. protocolo experimental y gráficas;
9. análisis teórico;
10. conclusiones y referencias;
11. enlace al historial de trabajo o bitácora incorporada.

Todo el cuaderno debe ejecutarse desde el inicio en un entorno limpio. Las celdas fuera de orden, dependencias implícitas y resultados pegados manualmente se consideran defectos reproducibles.

### Rúbrica

| Criterio | Peso |
|---|---:|
| Modelado, invariantes y auditoría de Hunt-and-Kill | 10% |
| Versión desde cero y estructuras de datos | 25% |
| Adaptación correcta a SimpleAI | 12% |
| Adaptación correcta a AIMA-Python | 13% |
| Pruebas, casos límite y concordancia | 15% |
| Experimentos, métricas y visualización | 12% |
| Análisis teórico y conclusiones | 8% |
| Defensa, trazabilidad y calidad del cuaderno | 5% |

Una solución que produzca un camino pero no satisfaga el contrato, las pruebas o la defensa no se considera completa.

## 14. Referencias de partida

- Russell, S. J. y Norvig, P. *Artificial Intelligence: A Modern Approach*, 4.ª edición, capítulo 3.
- SimpleAI, documentación de problemas y búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/docs/search_problems.rst>
- SimpleAI, implementación oficial de búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/simpleai/search/traditional.py>
- AIMA-Python, repositorio oficial: <https://github.com/aimacode/aima-python>
- AIMA-Python, módulo de búsqueda: <https://github.com/aimacode/aima-python/blob/master/aima/search.py>
- C. Y. Lee, “An Algorithm for Path Connections and Its Applications”, 1961.
- A. Adamatzky, “Physical maze solvers. All twelve prototypes implement 1961 Lee algorithm”, 2016: <https://arxiv.org/abs/1601.04672>

Las referencias orientan el estudio de las interfaces; no sustituyen la explicación de las decisiones tomadas.